In [2]:
import json
import pandas as pd

# === 1. Đọc file JSON ===
with open("job_7839872_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# === 2. Ánh xạ timeSet → tên khung giờ ===
time_set_map = {
    2: "Morning Peak",
    3: "Evening Peak",
    4: "Off Peak",
    5: "Weekend"
}

rows = []

# === 3. Duyệt tất cả các Feature ===
for feature in data.get("features", []):
    props = feature.get("properties", {})
    geom = feature.get("geometry", {})

    # ⚠️ Bỏ qua đối tượng metadata (Thu Duc Test Job)
    if geom is None or props.get("jobName") == "Thu Duc Test Job":
        continue

    coords = geom.get("coordinates") if geom else None
    segment_time_results = props.get("segmentTimeResults", [])

    # Lặp từng khung thời gian
    for result in segment_time_results:
        # Chuyển speedPercentiles thành chuỗi
        percentiles = result.get("speedPercentiles", [])
        percentiles_str = ",".join(map(str, percentiles)) if percentiles else ""

        # Tạo hàng dữ liệu
        row = {
            "segmentId": props.get("segmentId"),
            "newSegmentId": props.get("newSegmentId"),
            "streetName": props.get("streetName"),
            "speedLimit": props.get("speedLimit"),
            "frc": props.get("frc"),
            "distance": props.get("distance"),
            "timeSet": result.get("timeSet"),
            "timeSetName": time_set_map.get(result.get("timeSet"), "Unknown"),
            "dateRange": result.get("dateRange"),
            "harmonicAverageSpeed": result.get("harmonicAverageSpeed"),
            "medianSpeed": result.get("medianSpeed"),
            "averageSpeed": result.get("averageSpeed"),
            "standardDeviationSpeed": result.get("standardDeviationSpeed"),
            "travelTimeStandardDeviation": result.get("travelTimeStandardDeviation"),
            "sampleSize": result.get("sampleSize"),
            "averageTravelTime": result.get("averageTravelTime"),
            "medianTravelTime": result.get("medianTravelTime"),
            "travelTimeRatio": result.get("travelTimeRatio"),
            "speedPercentiles": percentiles_str,
            "coordinates": str(coords)
        }
        rows.append(row)

# === 4. Chuyển sang DataFrame ===
df = pd.DataFrame(rows)

# === 5. Ghi ra file CSV ===
output_file = "thu_duc_segments_filtered.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"✅ Đã tạo file CSV (bỏ qua Thu Duc Test Job): {output_file}")


✅ Đã tạo file CSV (bỏ qua Thu Duc Test Job): thu_duc_segments_filtered.csv
